In [15]:
import pandas as pd

df = pd.read_excel("../data/raw/customerChurn.xlsx")

df_clean = df.copy()

# print(df_clean.head())

In [16]:
print("Missing Total Charges:")
print(df_clean["Total Charges"].isna().sum())

print("\nBlank Total Charges:")
print(df_clean["Total Charges"].astype(str).str.strip().eq("").sum())

print("\nData type:")
print(df_clean["Total Charges"].dtype)

Missing Total Charges:
0

Blank Total Charges:
11

Data type:
object


Chuyen blank -> missing chuan

In [17]:
df_clean["Total Charges"] = pd.to_numeric(
    df_clean["Total Charges"],
    errors="coerce"
)

In [18]:
df_clean["Total Charges"].isna().sum()

np.int64(11)

fill = 0 doi voi cai data da xac minh 

In [19]:
mask = (
    df_clean["Total Charges"].isna()
    & (df_clean["Tenure Months"] == 0)
)

df_clean.loc[mask, "Total Charges"] = 0

In [20]:
print("Missing Total Charges:",
      df_clean["Total Charges"].isna().sum())

print("Total Charges dtype:",
      df_clean["Total Charges"].dtype)

print("Rows:",
      len(df_clean))

Missing Total Charges: 0
Total Charges dtype: float64
Rows: 7043


Luu clean_dataset

In [29]:
df_clean.to_csv(
    "../data/raw/customer_churn_clean.csv",
    index=False
)

### T014 — Data Cleaning Summary

Based on the Data Quality Report:

1. `Total Charges`
   - 11 blank values were identified.
   - All affected records had `Tenure Months = 0`.
   - Blank values were converted to missing values and replaced with 0
     only for customers with `Tenure Months = 0`.
   - The feature was converted from `object` to numerical (`float64`).

2. Duplicates
   - No duplicated records or duplicated CustomerIDs were detected.
   - No duplicate removal was required.

3. Outliers
   - No potential outliers were detected using the IQR method.
   - No outlier treatment was applied.

4. Class imbalance
   - No resampling was performed during data cleaning.
   - Class imbalance will be considered during modeling and evaluation.

The raw dataset was preserved unchanged.

In [31]:
print(df_clean.shape)
print(df_clean["Total Charges"].dtype)
print(df_clean["Total Charges"].isna().sum())
print(df_clean.duplicated().sum())
print(df_clean["CustomerID"].duplicated().sum())

(7043, 33)
float64
0
0
0


## T015 - Data Encoding

In [14]:
df_clean = pd.read_csv("../data/raw/customer_churn_clean.csv")
print(df_clean.head)

<bound method NDFrame.head of       CustomerID  Count        Country       State          City  Zip Code  \
0     3668-QPYBK      1  United States  California   Los Angeles     90003   
1     9237-HQITU      1  United States  California   Los Angeles     90005   
2     9305-CDSKC      1  United States  California   Los Angeles     90006   
3     7892-POOKP      1  United States  California   Los Angeles     90010   
4     0280-XJGEX      1  United States  California   Los Angeles     90015   
...          ...    ...            ...         ...           ...       ...   
7038  2569-WGERO      1  United States  California       Landers     92285   
7039  6840-RESVB      1  United States  California      Adelanto     92301   
7040  2234-XADUH      1  United States  California         Amboy     92304   
7041  4801-JZAZL      1  United States  California  Angelus Oaks     92305   
7042  3186-AJIEK      1  United States  California  Apple Valley     92308   

                    Lat Long   La

In [26]:
categorical_features = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method"
]

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

category_summary = pd.DataFrame({
    "Feature": categorical_features,
    "N_Unique": [
        df_clean[col].nunique()
        for col in categorical_features
    ],
    "Categories": [
        list(df_clean[col].unique())
        for col in categorical_features
    ]
})

category_summary

,Feature,N_Unique,Categories
0,Gender,2,"[Male, Female]"
1,Senior Citizen,2,"[No, Yes]"
2,Partner,2,"[No, Yes]"
3,Dependents,2,"[No, Yes]"
4,Phone Service,2,"[Yes, No]"
5,Multiple Lines,3,"[No, Yes, No phone service]"
6,Internet Service,3,"[DSL, Fiber optic, No]"
7,Online Security,3,"[Yes, No, No internet service]"
8,Online Backup,3,"[Yes, No, No internet service]"
9,Device Protection,3,"[No, Yes, No internet service]"


In [27]:
from sklearn.preprocessing import OneHotEncoder

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    drop="if_binary"
)

### Encoding Strategy

The dataset contains 16 categorical predictive features.

All selected categorical features are treated as nominal variables because
no clear ordinal relationship is assumed between their categories.

One-Hot Encoding is selected as the primary encoding strategy.

- Binary categorical features will use `drop="if_binary"` to avoid redundant
  dummy variables.
- Multi-category features will be represented using separate binary columns.
- `handle_unknown="ignore"` will be used to allow unseen categories during
  transformation.

The encoder will not be fitted on the complete dataset at this stage.
It will later be fitted using training data only as part of the preprocessing
pipeline to reduce the risk of data leakage.

Identifiers, target-related variables and potential leakage variables are
excluded from the encoding feature set.

# T016 — Numerical Feature Scaling

In [32]:
numerical_features = [
    "Tenure Months",
    "Monthly Charges",
    "Total Charges",
    "CLTV"
]

print(df_clean[numerical_features].describe().T)

                  count         mean          std      min      25%      50%  \
Tenure Months    7043.0    32.371149    24.559481     0.00     9.00    29.00   
Monthly Charges  7043.0    64.761692    30.090047    18.25    35.50    70.35   
Total Charges    7043.0  2279.734304  2266.794470     0.00   398.55  1394.55   
CLTV             7043.0  4400.295755  1183.057152  2003.00  3469.00  4527.00   

                     75%      max  
Tenure Months      55.00    72.00  
Monthly Charges    89.85   118.75  
Total Charges    3786.60  8684.80  
CLTV             5380.50  6500.00  


In [35]:
from sklearn.preprocessing import StandardScaler

numerical_scaler = StandardScaler()

### Scaling Strategy

The selected numerical predictive features have substantially different
scales. For example, `Tenure Months` ranges from 0 to 72, while
`Total Charges` and `CLTV` can reach several thousand.

StandardScaler is selected as the baseline scaling method.

No potential outliers were identified using the IQR method during Data
Understanding, so RobustScaler is not considered necessary at this stage.

The scaler will not be fitted on the complete dataset. It will be fitted
using training data only within the preprocessing pipeline to reduce the
risk of data leakage.

Tree-based models may not require feature scaling, but scaling is retained
in the preprocessing strategy for scale-sensitive algorithms.

# T017 — Feature Selection


In [36]:

leakage_features = ["Churn Label", "Churn Score", "Churn Reason"]
identifier_feature = "CustomerId"
constant_features = ["Count", "Contry", "State"]
location_features = ["City", "Zip Code", "Lat Long", "Latitude", "Longtitude"]

selected_features = [
    # Customer
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",

    # Relationship
    "Tenure Months",

    # Services
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",

    # Contract & Billing
    "Contract",
    "Paperless Billing",
    "Payment Method",
    "Monthly Charges",
    "Total Charges",

    # Customer value
    "CLTV"
]

target = "Churn Value"

In [37]:
print("Number of selected features:", len(selected_features))

for i, feature in enumerate(selected_features, 1):
    print(f"{i:2}. {feature}")

Number of selected features: 20
 1. Gender
 2. Senior Citizen
 3. Partner
 4. Dependents
 5. Tenure Months
 6. Phone Service
 7. Multiple Lines
 8. Internet Service
 9. Online Security
10. Online Backup
11. Device Protection
12. Tech Support
13. Streaming TV
14. Streaming Movies
15. Contract
16. Paperless Billing
17. Payment Method
18. Monthly Charges
19. Total Charges
20. CLTV


In [38]:
X  = df_clean[selected_features].copy()
y = df_clean[target].copy()

print(f"X shape: {X}")
print(f"y shape: {y}")

X shape:       Gender Senior Citizen Partner Dependents  Tenure Months Phone Service  \
0       Male             No      No         No              2           Yes   
1     Female             No      No        Yes              2           Yes   
2     Female             No      No        Yes              8           Yes   
3     Female             No     Yes        Yes             28           Yes   
4       Male             No      No        Yes             49           Yes   
5     Female             No     Yes         No             10           Yes   
6       Male            Yes      No         No              1            No   
7       Male             No      No         No              1           Yes   
8       Male             No     Yes        Yes             47           Yes   
9       Male             No     Yes         No              1            No   
10    Female             No      No         No             17           Yes   
11      Male             No      No        

### Feature Selection Summary

A total of 20 predictive features were selected for churn modeling.

The following groups were excluded:

- `CustomerID`: identifier without predictive meaning.
- `Count`, `Country`, and `State`: constant features with no discriminative value.
- `Churn Label`, `Churn Score`, and `Churn Reason`: excluded because they contain
  direct or post-outcome churn information and may introduce target leakage.
- Geographic variables (`City`, `Zip Code`, `Lat Long`, `Latitude`, `Longitude`)
  were excluded from the baseline model to reduce unnecessary complexity and
  high-cardinality location effects.

The target variable is `Churn Value`.

The resulting baseline feature matrix contains 20 predictive features.

# T018 — Train/Test Split

In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [41]:
print("Original:", X.shape)
print("Train:", X_train.shape)
print("Test:", X_test.shape)

Original: (7043, 20)
Train: (5634, 20)
Test: (1409, 20)


In [42]:
print("\nOriginal:")
print((y.value_counts(normalize=True) * 100).round(2))

print("\nTrain:")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTest:")
print((y_test.value_counts(normalize=True) * 100).round(2))


Original:
Churn Value
0    73.46
1    26.54
Name: proportion, dtype: float64

Train:
Churn Value
0    73.46
1    26.54
Name: proportion, dtype: float64

Test:
Churn Value
0    73.46
1    26.54
Name: proportion, dtype: float64


### Train/Test Split Strategy

The selected dataset was divided into training and testing sets using an
80/20 ratio.

A stratified split based on `Churn Value` was used because the target
distribution is imbalanced (approximately 73.46% non-churn and 26.54% churn).

`random_state=42` was used to make the split reproducible.

The test set will remain unseen during preprocessing fitting and model
training. Encoders and scalers will be fitted using training data only to
reduce the risk of data leakage.

In [44]:
X_train.to_csv("../data/raw/X_train.csv", index=False)
X_test.to_csv("../data/raw/X_test.csv", index=False)

y_train.to_csv("../data/raw/y_train.csv", index=False)
y_test.to_csv("../data/raw/y_test.csv", index=False)